# Detecção de Toxicidade com BERT — Projeto de Deep Learning

**Atividade Prática — Deep Learning para Demandas Reais**

Este notebook treina o modelo de **estado da arte** (BERT, um Transformer) para a frente de
**toxicidade** do projeto, usando a base **ToLD-Br**. Ele reproduz a metodologia documentada:
mesmo recorte binário, mesmo split estratificado e as mesmas métricas (macro-F1 como primária).

> **Antes de rodar:** menu `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU`.
> Depois, `Ambiente de execução → Executar tudo`. O treino leva poucos minutos numa GPU T4.


## 1. Instalar dependências e verificar a GPU

In [ ]:
!pip install -q -U transformers accelerate scikit-learn
import torch
print("GPU disponível:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Baixar a base ToLD-Br e montar o rótulo binário

In [ ]:
!git clone -q https://github.com/JAugusto97/ToLD-Br.git
import pandas as pd, numpy as np
SEED = 42
CATS = ["homophobia","obscene","insult","racism","misogyny","xenophobia"]
df = pd.read_csv("ToLD-Br/ToLD-BR.csv").drop_duplicates(subset=["text"]).reset_index(drop=True)
# tóxico se ALGUMA categoria recebeu >= 2 dos 3 votos (regra de maioria)
df["label"] = (df[CATS].max(axis=1) >= 2).astype(int)
print("Total:", len(df), "| tóxicos:", int(df.label.sum()), f"({df.label.mean()*100:.1f}%)")
df[["text","label"]].head()

## 3. Separação estratificada treino / validação / teste (70 / 15 / 15)

In [ ]:
from sklearn.model_selection import train_test_split
X = df["text"].astype(str).values; y = df["label"].values
Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=0.50, stratify=ytmp, random_state=SEED)
print("treino/val/teste:", len(ytr), len(yva), len(yte))

## 4. Tokenização (BERTabaporu — pré-treinado em tweets em português)

Usamos o **BERTabaporu** porque foi pré-treinado em ~237 milhões de tweets em português — mesmo
domínio (rede social) da base de toxicidade. Alternativa formal: `neuralmind/bert-base-portuguese-cased`.

In [ ]:
from transformers import AutoTokenizer
MODELO = "pablocosta/bertabaporu-base-uncased"
MAX_LEN = 128
tok = AutoTokenizer.from_pretrained(MODELO)

import torch
class DS(torch.utils.data.Dataset):
    def __init__(self, textos, labels):
        self.t = list(textos); self.y = list(labels)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        e = tok(self.t[i], truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in e.items()}
        item["labels"] = torch.tensor(self.y[i], dtype=torch.long)
        return item

ds_tr, ds_va, ds_te = DS(Xtr,ytr), DS(Xva,yva), DS(Xte,yte)

## 5. Modelo, perda ponderada e métricas

In [ ]:
import numpy as np
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, precision_recall_fscore_support, accuracy_score, roc_auc_score

# pesos de classe (desbalanceamento ~19% tóxico)
n = len(ytr); n1 = int(ytr.sum()); n0 = n - n1
pesos = torch.tensor([n/(2*n0), n/(2*n1)], dtype=torch.float)
print("pesos de classe [nao-tox, tox]:", pesos.tolist())

def compute_metrics(p):
    logits, labels = p
    preds = np.argmax(logits, axis=-1)
    prob = torch.softmax(torch.tensor(logits), dim=-1)[:,1].numpy()
    pr, rc, f1, _ = precision_recall_fscore_support(labels, preds, labels=[0,1], zero_division=0)
    return {"macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
            "accuracy": accuracy_score(labels, preds),
            "f1_tox": f1[1], "roc_auc": roc_auc_score(labels, prob)}

class TrainerPonderado(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(weight=pesos.to(out.logits.device))(out.logits, labels)
        return (loss, out) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(MODELO, num_labels=2)

## 6. Configuração de treino e fine-tuning

In [ ]:
# compatibilidade entre versoes do transformers (eval_strategy vs evaluation_strategy)
common = dict(output_dir="bert_tox", learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=3, weight_decay=0.01, warmup_ratio=0.1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    greater_is_better=True, logging_steps=50, report_to="none",
    fp16=torch.cuda.is_available(), seed=SEED)
try:
    args = TrainingArguments(eval_strategy="epoch", save_strategy="epoch", **common)
except TypeError:
    args = TrainingArguments(evaluation_strategy="epoch", save_strategy="epoch", **common)

trainer = TrainerPonderado(model=model, args=args,
    train_dataset=ds_tr, eval_dataset=ds_va, compute_metrics=compute_metrics)
trainer.train()

## 7. Avaliação no conjunto de teste

In [ ]:
import json
pred = trainer.predict(ds_te)
logits = pred.predictions
yhat = np.argmax(logits, axis=-1)
prob = torch.softmax(torch.tensor(logits), dim=-1)[:,1].numpy()
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
print(classification_report(yte, yhat, target_names=["nao-toxico","toxico"], digits=4))
macro = f1_score(yte, yhat, average="macro")
auc = roc_auc_score(yte, prob)
cm = confusion_matrix(yte, yhat)
print(f"macro-F1 = {macro:.4f} | ROC-AUC = {auc:.4f}")
res = {"modelo": MODELO, "macro_f1": round(float(macro),4), "roc_auc": round(float(auc),4),
       "cm": cm.tolist()}
json.dump(res, open("resultados_bert.json","w"), ensure_ascii=False, indent=2)
res

## 8. Gráficos: curva de loss (treino × validação) e matriz de confusão

In [ ]:
import matplotlib.pyplot as plt
hist = trainer.state.log_history
tr = [(h["epoch"], h["loss"]) for h in hist if "loss" in h and "epoch" in h]
ev = [(h["epoch"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(6,4))
if tr: plt.plot(*zip(*tr), label="loss treino", color="#5b4bdb")
if ev: plt.plot(*zip(*ev), "o-", label="loss validação", color="#0f9d72")
plt.xlabel("Época"); plt.ylabel("Loss"); plt.title("Curva de loss — BERT"); plt.legend(); plt.tight_layout()
plt.savefig("fig_loss_bert.png", dpi=150); plt.show()

plt.figure(figsize=(4.4,3.8)); plt.imshow(cm, cmap="Purples")
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i,j], ha="center", va="center", color="white" if cm[i,j]>cm.max()/2 else "black", fontsize=13)
plt.xticks([0,1],["nao-toxico","toxico"]); plt.yticks([0,1],["nao-toxico","toxico"])
plt.xlabel("Predito"); plt.ylabel("Real"); plt.title("Matriz de confusão — BERT")
plt.colorbar(); plt.tight_layout(); plt.savefig("fig_matriz_confusao_bert.png", dpi=150); plt.show()

## 9. Testar com texto novo (inferência)

In [ ]:
def prever(texto):
    model.eval()
    e = tok(texto, truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt").to(model.device)
    with torch.no_grad():
        p = torch.softmax(model(**e).logits, dim=-1)[0]
    rot = "TÓXICO" if p[1] > p[0] else "NÃO-TÓXICO"
    return f"{rot}  (confiança {max(p).item()*100:.1f}%)"

for t in ["você é um idiota completo, sai daqui",
          "bom dia pessoal, ótimo jogo ontem!",
          "esse comentário foi muito construtivo, obrigado"]:
    print(prever(t), "<-", t)

## 10. Salvar o modelo e baixar os resultados

In [ ]:
trainer.save_model("modelo_bert_toxicidade")
tok.save_pretrained("modelo_bert_toxicidade")
print("Modelo salvo em ./modelo_bert_toxicidade")
# baixar os artefatos (opcional)
try:
    from google.colab import files
    files.download("resultados_bert.json")
    files.download("fig_loss_bert.png")
    files.download("fig_matriz_confusao_bert.png")
except Exception as e:
    print("Para baixar, use o painel de Arquivos à esquerda. ", e)

---
## (Opcional) Frente política — inclinação ideológica (BERTimbau)

Para treinar a segunda frente, primeiro gere `discursos_camara.csv` na sua máquina com o script
`coletar_discursos_camara.py` do projeto, faça **upload** dele aqui (painel de Arquivos) e rode as
células abaixo. Use o **split agrupado por deputado** para evitar vazamento de identidade do autor.

In [ ]:
import os, pandas as pd, numpy as np, torch
if os.path.exists("discursos_camara.csv"):
    from sklearn.model_selection import GroupShuffleSplit
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
    dpol = pd.read_csv("discursos_camara.csv")
    dpol = dpol[dpol["ideologia"].isin(["esquerda","direita"])].copy()
    dpol["label"] = (dpol["ideologia"]=="direita").astype(int)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    tr_idx, te_idx = next(gss.split(dpol, dpol["label"], groups=dpol["id_deputado"]))
    print("treino/teste:", len(tr_idx), len(te_idx), "| classes:", dpol["label"].value_counts().to_dict())
    print("Modelo recomendado: neuralmind/bert-base-portuguese-cased (BERTimbau, texto formal).")
    print("Reaproveite a mesma estrutura de Dataset/Trainer das células 4-8, trocando MODELO e MAX_LEN=512.")
else:
    print("Suba 'discursos_camara.csv' para treinar a frente política (veja coletar_discursos_camara.py).")